# v19 — LGBM + ET Optuna Best Params

Apply Optuna-found hyperparameters from notebook 13 (100 Bayesian trials) to the full stacking pipeline.
These params were never incorporated into the stack — this tests whether they improve the Ridge-blended OOF RMSE.

**Changes vs v16 baseline ($21,552 OOF):**
- LGBM: Optuna 100-trial TPE best (individual val $21,518 vs $21,776 current)
- ET: Optuna 50-trial best (600 trees, max_depth 24 vs 300 trees, depth 20 current)
- XGB + CatBoost: unchanged

Only edit Cell 1 (CONFIG). Run all cells. Compare Cell 3 OOF RMSE vs v16 baseline $21,552.

In [1]:
# ── EDIT ONLY THIS CELL ────────────────────────────────────────────────────
CONFIG = {
    "experiment_name": "v19",  # Change per experiment — used as MLflow run label

    "data": {
        "train_path": "../../data/raw/train.csv",
        "test_path":  "../../data/raw/test.csv",
    },

    "features": {
        # OOF target encodings — same as v16 baseline
        "encode_pairs": [
            ("town",           "town_median_price",           1),
            ("postal_sector",  "postal_sector_median_price",  1),
            ("flat_model",     "flat_model_median_price",     1),
            ("town_flat_type", "town_flat_type_median_price", 3),
            ("Tranc_Year",     "year_median_price",           1),
        ],
        "drop_cols": [],
    },

    "models": {
        "active": ["xgb", "lgbm", "et", "catboost"],

        # XGB — unchanged from v16 (Optuna individual gain was marginal vs stack risk)
        "xgb": {
            "n_estimators": 2000, "max_depth": 7, "learning_rate": 0.05,
            "subsample": 0.9, "colsample_bytree": 0.4, "min_child_weight": 7,
            "reg_alpha": 0.01, "reg_lambda": 1.5,
            "random_state": 42, "n_jobs": -1, "verbosity": 0, "tree_method": "hist",
        },

        # LGBM — Optuna TPE 100-trial best (nb 13): individual val RMSE $21,518 vs $21,776 current
        "lgbm": {
            "n_estimators": 888, "max_depth": 14, "num_leaves": 317,
            "learning_rate": 0.0272, "subsample": 0.9225, "colsample_bytree": 0.4564,
            "min_child_samples": 31, "reg_alpha": 0, "reg_lambda": 1.135,
            "random_state": 42, "n_jobs": -1, "verbosity": -1,
        },

        # ET — Optuna 50-trial best (nb 13): 600 trees, depth 24 vs 300 trees, depth 20 current
        "et": {
            "n_estimators": 600, "min_samples_split": 7, "min_samples_leaf": 2,
            "max_features": 0.829, "max_depth": 24,
            "random_state": 42, "n_jobs": -1,
        },

        # CatBoost — unchanged from v16 (never tuned; Phase 2 handles this)
        "catboost": {
            "iterations": 1000, "depth": 7, "learning_rate": 0.08,
            "l2_leaf_reg": 1, "colsample_bylevel": 0.7, "min_child_samples": 5,
            "loss_function": "RMSE", "random_seed": 42, "verbose": 0,
        },

        "ridge_alphas": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    },

    "cv": {"n_splits": 5, "random_state": 42},

    "output": {
        "submission_path": "../../outputs/submissions/sub_v19.csv",
        # "fulldata_submission_path": "../../outputs/submissions/sub_v19_fulldata.csv",
        "sample_path": "../../outputs/submissions/sample_sub_reg.csv",
    },
}

In [2]:
import sys
sys.path.insert(0, "../../")

from src.pipeline import run_pipeline

results = run_pipeline(CONFIG)

/Users/tehmenghai/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/04/29 14:47:00 INFO mlflow.tracking.fluent: Experiment with name 'v19' does not exist. Creating a new experiment.



Experiment: v19
Loaded — Train: (150634, 77)  |  Test: (16737, 76)
After engineering — Train: (150634, 100)  |  Test: (16737, 99)
OOF encodings applied: ['town_median_price', 'postal_sector_median_price', 'flat_model_median_price', 'town_flat_type_median_price', 'year_median_price']
Feature matrix: 78 columns

Running 5-fold stacking...
 Fold  XGB-RMSE        LGBM-RMSE       ET-RMSE         CATBOOST-RMSE 
-----------------------------------------------------------------------
    1  $       21,510  $       21,418  $       22,932  $       22,255
    2  $       22,278  $       22,254  $       23,610  $       23,062
    3  $       21,643  $       21,578  $       23,157  $       22,600
    4  $       21,899  $       21,860  $       23,396  $       22,565
    5  $       21,777  $       21,693  $       23,241  $       22,556
-----------------------------------------------------------------------
 Mean  $       21,821  $       21,761  $       23,267  $       22,607

Ridge alpha sweep:
     a

In [3]:
print(f'OOF RMSE:      ${results["oof_rmse"]:,.0f}')
print(f'Ridge weights: {results["weights"]}')
print(f'Best alpha:    {results["best_alpha"]}')
print()
print('Per-model mean OOF RMSE:')
import numpy as np
for m, scores in results['fold_scores'].items():
    print(f'  {m:<12} ${np.mean(scores):>10,.0f}')
print()
print('To view all experiments in MLflow UI:')
print('  cd <project_root>')
print('  mlflow ui --backend-store-uri sqlite:///mlflow.db')
print('  Open http://localhost:5000')

OOF RMSE:      $21,533
Ridge weights: {'xgb': 0.3363775317299665, 'lgbm': 0.3427678373242877, 'et': 0.1521950391100853, 'catboost': 0.1713897345748053}
Best alpha:    0.001

Per-model mean OOF RMSE:
  xgb          $    21,821
  lgbm         $    21,761
  et           $    23,267
  catboost     $    22,607

To view all experiments in MLflow UI:
  cd <project_root>
  mlflow ui --backend-store-uri sqlite:///mlflow.db
  Open http://localhost:5000
